In [ ]:
from dotenv import load_dotenv
from typing import TypedDict, Literal

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()

In [ ]:
from dotenv import load_dotenv
from typing import TypedDict, Literal

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()

In [ ]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [ ]:
def generate_tweet(state: TweetState) -> dict:
    messages = [
        SystemMessage(
            content="You are a funny, clever Twitter/X writer. Write original, concise posts."
        ),
        HumanMessage(
            content=f"""Write a short, original, engaging tweet about: {state['topic']}

Rules:
- Do not use question-and-answer format.
- Stay under 280 characters.
- Prefer observational humor, irony, relatable takes, or a sharp insight.
- Use simple, natural English.
- Return only the tweet text."""
        ),
    ]

    response = generator_llm.invoke(messages).content
    return {"tweet": str(response).strip()}

In [ ]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        description="Whether the tweet meets the quality criteria."
    )
    feedback: str = Field(
        description="A concise explanation of strengths and weaknesses, including actionable improvements."
    )

structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [ ]:
def evaluate_tweet(state: TweetState) -> dict:
    messages = [
        SystemMessage(
            content=(
                "You are a strict editor evaluating Twitter/X posts for originality, "
                "humor, punchiness, shareability, and format."
            )
        ),
        HumanMessage(
            content=f"""Evaluate this tweet for the topic: {state['topic']}

Tweet:
{state['tweet']}

Criteria:
1. Originality: does it feel fresh rather than generic?
2. Humor or insight: is it genuinely amusing, clever, or useful for the topic?
3. Punchiness: is it concise and memorable?
4. Shareability: might people want to repost it?
5. Format: is it under 280 characters and not a question-and-answer joke?

Mark it "approved" only if it meets the criteria reasonably well.
Otherwise mark it "needs_improvement" and give specific, actionable feedback."""
        ),
    ]

    result = structured_evaluator_llm.invoke(messages)
    return {
        "evaluation": result.evaluation,
        "feedback": result.feedback,
    }

In [ ]:
def optimize_tweet(state: TweetState) -> dict:
    next_iteration = state["iteration"] + 1

    messages = [
        SystemMessage(
            content=(
                "You improve short Twitter/X posts using editorial feedback. "
                "Keep the strongest idea, fix the stated weaknesses, and write naturally."
            )
        ),
        HumanMessage(
            content=f"""Topic: {state['topic']}

Current tweet:
{state['tweet']}

Evaluator feedback:
{state['feedback']}

Rewrite the tweet to address the feedback.
Rules:
- Keep it under 280 characters.
- Do not use question-and-answer format.
- Avoid generic filler and weak endings.
- Return only the revised tweet text."""
        ),
    ]

    response = optimizer_llm.invoke(messages).content
    return {
        "tweet": str(response).strip(),
        "iteration": next_iteration,
    }

In [ ]:
def route_evaluation(state: TweetState) -> Literal["approved", "needs_improvement", "max_iterations"]:
    if state["evaluation"] == "approved":
        return "approved"

    if state["iteration"] >= state["max_iteration"]:
        return "max_iterations"

    return "needs_improvement"

In [ ]:
graph = StateGraph(TweetState)

graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")

graph.add_conditional_edges(
    "evaluate",
    route_evaluation,
    {
        "approved": END,
        "needs_improvement": "optimize",
        "max_iterations": END,
    },
)

graph.add_edge("optimize", "evaluate")

workflow = graph.compile()
workflow

In [ ]:
initial_state: TweetState = {
    "topic": "software developers and debugging",
    "tweet": "",
    "evaluation": "needs_improvement",
    "feedback": "",
    "iteration": 0,
    "max_iteration": 3,
}

result = workflow.invoke(initial_state)

print("Topic:", result["topic"])
print("\\nFinal tweet:", result["tweet"])
print("\\nEvaluation:", result["evaluation"])
print("\\nFeedback:", result["feedback"])
print(f"Optimization iterations: {result['iteration']} / {result['max_iteration']}")
if result["evaluation"] != "approved":
    print("Stopped because the maximum optimization iterations were reached.")